# Setup 

## Random seed for notebook


In [25]:
my_random_seed = 123
k_folds = 5

## Libraries

In [26]:
# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


# Data

The "machine learning ready" dataframe is the one produced by the previous notebook (`step01_imputation_and_dim_reduction.ipynb`).


## ML ready dataframe

In [27]:
ml_df = pd.read_csv("ml_ready/genotypes_reduced_and_pheno.tsv", sep="\t", index_col=0)
print("Original data shape:", ml_df.shape)
print("Number of PCs:", ml_df.shape[1] - 1)
print("Number of samples:", ml_df.shape[0])
ml_df.head( n=3)

Original data shape: (187, 188)
Number of PCs: 187
Number of samples: 187


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC179,PC180,PC181,PC182,PC183,PC184,PC185,PC186,PC187,blues
IVIP100034,54.549141,38.133582,142.660935,-49.956386,0.748640,5.414148,10.059607,17.291886,10.684848,-13.075276,...,-0.488115,-7.384926,8.736871,-63.275841,-71.029276,-0.929247,0.059199,-0.034280,-3.886158e-13,41.743707
IVIP100081,-46.480913,-36.233098,5.090303,2.873656,0.927594,-8.819168,27.713060,-26.812522,-73.067061,-21.773864,...,-1.161345,0.985076,-95.207466,-7.708934,-3.529448,-0.040486,0.040201,-0.013806,-3.886158e-13,37.393715
IVIP100121,-45.764929,-32.709310,4.784485,2.183271,-1.832020,-3.176112,11.474953,-3.396201,-9.216511,-1.973546,...,0.928534,0.013973,-2.558860,-0.443817,0.194822,-0.054120,-0.003396,0.010738,-3.886158e-13,40.923005


## Target and features

In [28]:
# target
y = ml_df["blues"]
# features
X = ml_df.drop(columns=["blues"])

# Gradient Boosting Regressor model

## Initialize

In [29]:
my_gbr_params = {
    "n_estimators": 1000,
    "max_depth": None, # If None, then nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples
    "min_samples_split": 2,
    "max_features": "sqrt",  # The number of features to consider when looking for the best split
    "learning_rate": 0.01,
    "loss": "squared_error",
    # add early stopping parameters
    "validation_fraction": 0.1,
    "n_iter_no_change": 10
}

reg = GradientBoostingRegressor(**my_gbr_params, random_state=my_random_seed)

Create dataframes to:
1. hold model results,  
2. feature importances computed through permutations, 
3. and feature importances 

## Fit model with k-fold cross validation 

In [ ]:
# k-fold cross-validation using train, val and test of 0.7, 0.1, 0.2 respectively
for k in range(k_folds):
    print(f"Fold {k+1}/{k_folds}")
    # 70% for training
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=my_random_seed + k
    )
    # 10% for validation and 20% for testing
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.2, random_state=my_random_seed + k
    )
    reg.fit(X_train, y_train)

    # Compute MSE for training, validation, and test sets
    y_train_pred = reg.predict(X_train)
    mse_train = mean_squared_error(y_train, y_train_pred)
    y_val_pred = reg.predict(X_val)
    mse_val = mean_squared_error(y_val, y_val_pred)
    y_pred = reg.predict(X_test)
    mse_test = mean_squared_error(y_test, y_pred)
    # Add results to the results dataframe for fold k
    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame({"fold": k + 1, "mse_train": mse_train, "mse_val": mse_val, "mse_test": mse_test}, index=[0])
        ]
    )

    print(f"Mean Squared Error: {mse_test:.4f}")


In [ ]:
def run_kfold_cross_validation(X, y, 
                               model_params, 
                               k_folds=5, 
                               random_seed=123, 
                               train_size=0.7, val_size=0.1, test_size=0.2):
    """
    Run k-fold cross-validation for gradient boosting regression.
    
    Parameters:
    -----------
    X : pd.DataFrame
        Feature matrix
    y : pd.Series
        Target variable
    model_params : dict
        Parameters for GradientBoostingRegressor
    k_folds : int, default=5
        Number of folds for cross-validation
    random_seed : int, default=123
        Random seed for reproducibility
    train_size : float, default=0.7
        Proportion of data for training
    val_size : float, default=0.1
        Proportion of data for validation
    test_size : float, default=0.2
        Proportion of data for testing
        
    Returns:
    --------
    all_results : dict
        Dictionary with:
        - "perm_importance_df": Dataframe with permutation importance scores for each fold
        - "results_df": Dataframe with original MSE scores for each fold (to calculate original mean and std model scores)
        - "permuted_model_scores": List of model scores on permuted data for each fold (to compute model p-value)
    """
    
    # Initialize results dataframe
    results_df = pd.DataFrame(columns=["fold", "mse_train", "mse_val", "mse_test"])
    
    # k-fold cross-validation
    for k in range(k_folds):
        print(f"Fold {k+1}/{k_folds}")
        
        # First split: separate training data from temp data
        temp_size = val_size + test_size
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=temp_size, random_state=random_seed + k
        )
        
        # Second split: separate validation from test data
        test_size_adjusted = test_size / temp_size  # Adjust test_size relative to temp data
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=test_size_adjusted, random_state=random_seed + k
        )
        
        # Create and fit model for this fold
        reg = GradientBoostingRegressor(**model_params, random_state=random_seed + k)
        reg.fit(X_train, y_train)
        #=============================================================== 
        # Compute MSE for training, validation, and test sets
        y_train_pred = reg.predict(X_train)
        mse_train = mean_squared_error(y_train, y_train_pred)
        y_val_pred = reg.predict(X_val)
        mse_val = mean_squared_error(y_val, y_val_pred)
        y_pred = reg.predict(X_test)
        mse_test = mean_squared_error(y_test, y_pred)
        
        #======
        # Compute variable importance using permutation importance
        perm_importance = permutation_importance(reg, X_test, y_test, n_repeats=10, random_state=random_seed + k)
        perm_importance_df_fold = pd.DataFrame({
            "feature": X.columns,
            "importance_mean": perm_importance.importances_mean,
            "importance_std": perm_importance.importances_std,
            "fold": k + 1
        })
        perm_importance_df = pd.concat([perm_importance_df, perm_importance_df_fold], ignore_index=True)    

        # Add results to the results dataframe for fold k
        fold_results = pd.DataFrame({
            "fold": [k + 1], 
            "mse_train": [mse_train], 
            "mse_val": [mse_val], 
            "mse_test": [mse_test]
        })
        results_df = pd.concat([results_df, fold_results], ignore_index=True)

    all_results = {
        "results_df": results_df,
        "perm_importance_df": perm_importance_df
    }        
    return all_results

In [34]:
results_df

,fold,mse_train,mse_val,mse_test
0,1,17.451266,20.862128,31.902345
0,2,17.369909,23.803960,18.925629
0,3,16.982810,18.366390,42.851833
0,4,16.375273,17.378475,22.340319
0,5,20.348702,13.538056,14.265759


## Metrics and feature importance

In [ ]:
# Score model metrics 


# Tutorial

## Implementation with xgboost

In [ ]:
# # First XGBoost model for Pima Indians dataset
# from numpy import loadtxt
# from xgboost import XGBClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score
# # load data
# dataset = loadtxt('pima-indians-diabetes.csv', delimiter=",")
# # split data into X and y
# X = dataset[:,0:8]
# Y = dataset[:,8]
# # split data into train and test sets
# test_size = 0.33
# X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=test_size, random_state=my_random_seed)
# # fit model no training data
# model = XGBClassifier()
# model.fit(X_train, y_train)
# # make predictions for test data
# y_pred = model.predict(X_test)
# predictions = [round(value) for value in y_pred]
# # evaluate predictions
# accuracy = accuracy_score(y_test, predictions)
# print("Accuracy: %.2f%%" % (accuracy * 100.0))

## Implementation with scikit-learn

In [ ]:
# diabetes = datasets.load_diabetes()
# X, y = diabetes.data, diabetes.target

# my_test_size = 0.1
    
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=my_test_size, random_state=my_random_seed
# )

# params = {
#     "n_estimators": 500,
#     "max_depth": 4,
#     "min_samples_split": 5,
#     "learning_rate": 0.01,
#     "loss": "squared_error",
#     # add early stopping parameters
#     "validation_fraction": 0.1,
#     "n_iter_no_change": 10
# }

In [ ]:
# reg = ensemble.GradientBoostingRegressor(**params)
# reg.fit(X_train, y_train)

# mse = mean_squared_error(y_test, reg.predict(X_test))
# print("The mean squared error (MSE) on test set: {:.4f}".format(mse))

# # extract r2 score
# r2 = reg.score(X_test, y_test)
# print("The R2 score on test set: {:.4f}".format(r2))

# # plot y_test vs y_predicted
# y_pred = reg.predict(X_test)
# plt.figure(figsize=(8, 8))
# plt.scatter(y_test, y_pred, c="crimson")
# plt.yscale("log")
# plt.xscale("log")
# p1 = max(max(y_pred), max(y_test))
# p2 = min(min(y_pred), min(y_test))
# plt.plot([p1, p2], [p1, p2], 'b-')
# plt.xlabel('True Values', fontsize=15)
# plt.ylabel('Predictions', fontsize=15)
# plt.axis('equal')
# plt.show()